In [1]:
!git clone https://github.com/vc64/ultralytics.git
%cd ultralytics
!pip install -e .
%cd ..

Cloning into 'ultralytics'...
remote: Enumerating objects: 51979, done.
remote: Total 51979 (delta 0), reused 0 (delta 0), pack-reused 51979 (from 1)
Receiving objects: 100% (51979/51979), 33.37 MiB | 28.86 MiB/s, done.
Resolving deltas: 100% (37962/37962), done.
/home/user/ultralytics
Obtaining file:///home/user/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 74.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 29.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import sys
sys.path.insert(0, '/home/user/ultralytics')

from ultralytics.nn import tasks

In [2]:
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
# import timm
import torch
import torch.nn as nn

In [4]:
from ultralytics.data.utils import check_det_dataset

check_det_dataset("VisDrone.yaml")


WARNING ⚠️ Dataset 'VisDrone.yaml' images not found, missing path '/home/user/datasets/VisDrone/images/val'
Unzipping /home/user/datasets/VisDrone/VisDrone2019-DET-val.zip to /home/user/datasets/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.0Kfiles/s 1.1s0.1MB/s 1.5s<6.7s
Unzipping /home/user/datasets/VisDrone/VisDrone2019-DET-test-dev.zip to /home/user/datasets/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.5Kfiles/s 2.2s6s<4.4s
Unzipping /home/user/datasets/VisDrone/VisDrone2019-DET-train.zip to /home/user/datasets/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 1.8Kfiles/s 7.2ss<0.0s
Converting train: ━━━━━━━━━━━━ 6471 2.3Kit/s 2.9s
Converting val: ━━━━━━━━━━━━ 548 1.1Kit/s 0.3ss
Converting test: ━━━━━━━━━━━━ 1610 2.2Kit/s 0.7s
Dataset download success ✅ (19.7s), saved to /home/user/datasets



{'path': PosixPath('/home/user/datasets/VisDrone'),
 'train': '/home/user/datasets/VisDrone/images/train',
 'val': '/home/user/datasets/VisDrone/images/val',
 'test': '/home/user/datasets/VisDrone/images/test',
 'names': {0: 'pedestrian',
  1: 'people',
  2: 'bicycle',
  3: 'car',
  4: 'van',
  5: 'truck',
  6: 'tricycle',
  7: 'awning-tricycle',
  8: 'bus',
  9: 'motor'},
 'download': 'import os\nfrom pathlib import Path\nimport shutil\n\nfrom ultralytics.utils.downloads import download\nfrom ultralytics.utils import ASSETS_URL, TQDM\n\n\ndef visdrone2yolo(dir, split, source_name=None):\n    """Convert VisDrone annotations to YOLO format with images/{split} and labels/{split} structure."""\n    from PIL import Image\n\n    source_dir = dir / (source_name or f"VisDrone2019-DET-{split}")\n    images_dir = dir / "images" / split\n    labels_dir = dir / "labels" / split\n    labels_dir.mkdir(parents=True, exist_ok=True)\n\n    # Move images to new structure\n    if (source_images_dir := s

In [3]:
from huggingface_hub import hf_hub_download

def load_teacher():
    model_path = hf_hub_download(
        repo_id="kailunw/visdrone-yolo26m",
        filename="best.pt"
    )
    
    model = YOLO(model_path)
    return model

In [4]:
class MGDBlock(nn.Module):
    def __init__(self, student_channels, teacher_channels, mask_ratio=0.5):
        super().__init__()
        self.mask_ratio = mask_ratio
        # project student channels up to teacher channels
        # self.align = nn.Conv2d(student_channels, teacher_channels, kernel_size=1)
        
        # generation process as described for MGD
        self.generation = nn.Sequential(
            nn.Conv2d(teacher_channels, teacher_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(teacher_channels, teacher_channels, kernel_size=3, padding=1)
        )

    def forward(self, student_feat, teacher_feat):
        # s_aligned = self.align(student_feat)
        s_aligned = student_feat
        B, C, H, W = s_aligned.shape
        device = s_aligned.device
        
        # generate random mask
        mask = (torch.rand((B, 1, H, W), device=device) > self.mask_ratio).float()
        s_masked = s_aligned * mask
        
        # compute mse between teacher (orig input) and student feats (masked input)
        s_generated = self.generation(s_masked)
        return F.mse_loss(s_generated, teacher_feat)

In [5]:
import torch
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

class KDDetectionTrainer(DetectionTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        self.teacher = load_teacher().model.eval()
        for param in self.teacher.parameters():
            param.requires_grad = False
        self.original_loss_fn = None

        self.teacher_feats = {}
        self.student_feats = {}

        self.student_backbone_layers = [13, 14, 15]
        self.teacher_backbone_layers = [3, 5, 7]

        self.student_channels = [256, 512, 512]
        self.teacher_channels = [256, 512, 512]
        self.student_handles = None
    
    def _register_hooks(self, model, target_dict, layer_indices):
        """Helper function to attach hooks to specific layers."""
        handles = []
        def get_hook(idx):
            def hook_fn(module, inp, out):
                target_dict[idx] = out
            hook_fn.is_distillation_hook = True
            return hook_fn
            
        for i, layer_idx in enumerate(layer_indices):
            handle = model.model[layer_idx].register_forward_hook(get_hook(i))
            handles.append(handle)
        
        return handles
    
    def _remove_hooks(self, model, layer_indices):
        """Delete hooks marked as distillation hooks on the given model"""
        for layer_idx in layer_indices:
            layer = model.model[layer_idx]
            keys_to_remove = [
                hook_id for hook_id, hook in layer._forward_hooks.items() 
                if getattr(hook, 'is_distillation_hook', False)
            ]
            for k in keys_to_remove:
                del layer._forward_hooks[k]

    def get_model(self, cfg=None, weights=None, verbose=True):
        student_model = super().get_model(cfg, weights, verbose)
        self.original_loss_fn = student_model.loss

        self._register_hooks(
            self.teacher, 
            self.teacher_feats, 
            self.teacher_backbone_layers
        )
        self.student_handles = self._register_hooks(
            student_model, 
            self.student_feats, 
            self.student_backbone_layers
        )

        mgd_blocks = nn.ModuleList([
            MGDBlock(self.student_channels[0], self.teacher_channels[0]),
            MGDBlock(self.student_channels[1], self.teacher_channels[1]),
            MGDBlock(self.student_channels[2], self.teacher_channels[2])
        ])
        device = next(student_model.model.parameters()).device
        student_model.mgd_blocks = mgd_blocks.to(device)

        def custom_kd_loss(batch, preds=None):
            batch_size = batch["img"].shape[0]
            if preds is None:
                student_preds = student_model(batch["img"])
            else:
                student_preds = preds
            
            hard_loss, loss_items = self.original_loss_fn(batch, student_preds)

            if type(student_preds) is tuple:
                student_preds = student_preds[1]
            student_boxes, student_scores = student_preds["boxes"], student_preds["scores"]
            
            self.teacher.to(device=batch["img"].device, dtype=batch["img"].dtype)
            with torch.no_grad():
                # need to set just detection head to training mode to get more info for distillation
                teacher_head = self.teacher.model[-1] 
                teacher_head.training = True
                teacher_preds = self.teacher(batch["img"])["one2many"]
                teacher_head.training = False
            
            mgd_loss = 0.0
            for i in range(3):
                # get features recorded from hooks
                s_feat = self.student_feats[i]
                t_feat = self.teacher_feats[i]
                mgd_loss += student_model.mgd_blocks[i](s_feat, t_feat)

            # avoid memory leakage
            self.student_feats.clear()
            self.teacher_feats.clear()

            teacher_boxes, teacher_scores = teacher_preds["boxes"], teacher_preds["scores"]

            # classification loss
            temperature = 3.0
            # since KL div from torch needs first dist to be log
            s_log_probs = F.log_softmax(student_scores / temperature, dim=-1)
            t_probs = F.softmax(teacher_scores / temperature, dim=-1)
            class_loss = F.kl_div(s_log_probs, t_probs, reduction='batchmean') * (temperature ** 2)

            # bounding box loss
            B, _, N = student_boxes.shape
            student_boxes = student_boxes.view(B, 4, 16, N)
            student_box_probs = student_boxes.softmax(dim=2)
            bins = torch.arange(16, device=student_box_probs.device).view(1, 1, 16, 1)

            decoded_student_boxes = (student_box_probs * bins).sum(dim=2)
            box_loss = F.mse_loss(decoded_student_boxes, teacher_boxes)

            # combine output losses, using weights to try and even out scales
            soft_loss = torch.stack([
                box_loss * 0.06,
                class_loss * 0.02,
                mgd_loss
            ])
            # print(hard_loss)
            # print(soft_loss)

            # total loss, weighted avg of original loss and distillation loss
            alpha = 0.5
            total_loss = (alpha * hard_loss) + ((1 - alpha) * soft_loss * batch_size)
            output_loss_items = (alpha * loss_items) + ((1 - alpha) * soft_loss.detach())
            return total_loss, output_loss_items

        # replace loss func of student with our custom loss        
        student_model.loss = custom_kd_loss
        return student_model
    
    def save_model(self):
        """Override save_model to detach unpicklable local loss func and forward hooks"""
        custom_loss = self.model.loss
        self.model.loss = self.original_loss_fn

        # remove forward hooks
        self._remove_hooks(self.model, self.student_backbone_layers)
        
        # also need to detach exponential moving avg loss, saved by ultralytics
        custom_ema_loss = None
        if hasattr(self, 'ema') and self.ema is not None:
            custom_ema_loss = self.ema.ema.loss
            self.ema.ema.loss = self.original_loss_fn
            self._remove_hooks(self.ema.ema, self.student_backbone_layers)

        # # remove forward hooks
        # print(self.student_handles)
        # for handle in self.student_handles:
        #     handle.remove()
        
        super().save_model()
        
        self.model.loss = custom_loss
        if custom_ema_loss is not None:
            self.ema.ema.loss = custom_ema_loss
        
        self.student_handles = self._register_hooks(
            self.model, self.student_feats, self.student_backbone_layers
        )

        if hasattr(self, 'ema') and self.ema is not None:
            self.ema_hook_handles = self._register_hooks(
                self.ema.ema, self.student_feats, self.student_backbone_layers
            )

In [6]:
curriculum_stages = [
    {"data": "visdrone_stage1_easy.yaml", "epochs": 20},
    {"data": "visdrone_stage2_med.yaml",  "epochs": 50},
    {"data": "visdrone_stage3_full.yaml", "epochs": 130}
]

current_weights = "mobilenetv4-yolo26m.yaml"
for stage_idx, stage_cfg in enumerate(curriculum_stages):
    stage_num = stage_idx + 1
    args = {
        "model": current_weights,
        "data": stage_cfg["data"],
        "epochs": stage_cfg["epochs"],
        "batch": 16,
        "imgsz": 1024,
        "patience": 50,
        "amp": True,
        "project": "kd_curriculum",
        "name": f"stage_{stage_num}",
    }
    
    trainer = KDDetectionTrainer(overrides=args)
    trainer.train()

    current_weights = f"ultralytics/runs/detect/kd_curriculum/stage_{stage_num}/weights/best.pt"

Ultralytics 8.4.45 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (Tesla V100-SXM3-32GB, 32494MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=visdrone_stage1_easy.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mobilenetv4-yolo26m.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=stage_1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m


                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2, 1]              
  1                  -1  1      9280  ultralytics.nn.modules.conv.Conv             [32, 32, 3, 2, 1]             
  2                  -1  1      1088  ultralytics.nn.modules.conv.Conv             [32, 32, 1, 1]                
  3                  -1  1     27840  ultralytics.nn.modules.conv.Conv             [32, 96, 3, 2, 1]             
  4                  -1  1      6272  ultralytics.nn.modules.conv.Conv             [96, 64, 1, 1]                
  5                  -1  1     38208  ultralytics.nn.tasks.UniversalInvertedBlock  [64, 96, 5, 2, 3.0, True, True]
  6                  -1  2     79104  ultralytics.nn.tasks.UniversalInvertedBlock  [96, 96, 3, 1, 2.0, False, True]
  7                  -1  1     75744  ultralytics.nn.tasks.UniversalInvertedBlock  [

KeyboardInterrupt: 

In [5]:
best_model = YOLO("ultralytics/runs/detect/kd_curriculum/stage_3/weights/best.pt")
best_model.val(data="ultralytics/cfg/datasets/VisDrone.yaml", split="test")

Ultralytics 8.4.45 🚀 Python-3.12.3 torch-2.6.0+cu124 CPU (Intel Xeon Platinum 8168 CPU @ 2.70GHz)
mobilenetv4-YOLO26m summary: 193 layers, 23,277,774 parameters, 0 gradients, 32.2 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.5 ms, read: 79.3±28.2 MB/s, size: 134.5 KB)
val: Scanning /home/user/datasets/VisDrone/labels/test... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 1.2Kit/s 1.4s0.0s
val: New cache created: /home/user/datasets/VisDrone/labels/test.cache


/home/user/.project/lib/python3.12/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 5.4s/it 9:027.4s
                   all       1610      75102      0.504      0.388      0.359      0.206
            pedestrian       1197      21006      0.535      0.363      0.355      0.143
                people        797       6376       0.53      0.193      0.198      0.071
               bicycle        377       1302      0.355      0.175      0.161     0.0672
                   car       1530      28074      0.724      0.768      0.757      0.485
                   van       1168       5771      0.472      0.416      0.375      0.253
                 truck        750       2659      0.468       0.42      0.375      0.243
              tricycle        245        530      0.307      0.358      0.236      0.128
       awning-tricycle        233        599       0.44      0.245      0.206      0.128
                   bus        838       2940      0.728      0.501

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d5dddc0c5c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0